# KAP-FinQA-TR — Ana Pipeline
**Notebook 1/4:** Kurulum → Zero-Shot Değerlendirme → QLoRA Fine-Tuning → Fine-Tuned Değerlendirme

Bu notebook Qwen2.5-1.5B ve TinyLlama-1.1B için tam pipeline'ı çalıştırır.  
SmolLM2-360M ve Gemma-4-E2B için Notebook 2'ye bakın.

**Gereksinimler:** Google Colab T4 GPU, Google Drive'da `/kap_finqa/` klasörü  
**Tahmini süre:** Qwen ~2.5 sa | TinyLlama ~1.3 sa

In [ ]:
# HÜCRE 1 — Kurulum
!pip install -q transformers peft bitsandbytes trl datasets \
             accelerate rouge-score nltk chromadb \
             sentence-transformers huggingface_hub pymupdf

In [ ]:
# HÜCRE 2 — Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/kap_finqa')
print(os.getcwd())
!ls

In [ ]:
# HÜCRE 3 — Scriptleri /content'e kopyala
import shutil, os
scripts = ['1_zero_shot_eval.py', '2_finetune.py', '3_eval_finetuned.py',
           '4_rag_pipeline.py', '5_upload_hf.py']
for s in scripts:
    src = f'/content/drive/MyDrive/kap_finqa/{s}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{s}')
        print(f'✅ {s} kopyalandı')
    else:
        print(f'❌ {s} bulunamadı — Drive konumunu kontrol edin')

## Zero-Shot Değerlendirme

In [ ]:
# HÜCRE 4 — Zero-Shot: Qwen2.5-1.5B
!python /content/1_zero_shot_eval.py \
    --model Qwen2.5-1.5B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --samples 331

In [ ]:
# HÜCRE 5 — Zero-Shot: TinyLlama-1.1B
!python /content/1_zero_shot_eval.py \
    --model TinyLlama-1.1B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --samples 331

In [ ]:
# HÜCRE 6 — Zero-Shot: Mistral-7B (referans model)
!python /content/1_zero_shot_eval.py \
    --model Mistral-7B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv \
    --samples 50

## QLoRA Fine-Tuning

In [ ]:
# HÜCRE 7 — Fine-Tune: Qwen2.5-1.5B (seed=42, ~2.5 saat)
!python /content/2_finetune.py \
    --model Qwen2.5-1.5B \
    --train_csv /content/drive/MyDrive/kap_finqa/train.csv \
    --val_csv /content/drive/MyDrive/kap_finqa/validation.csv \
    --seed 42

In [ ]:
# HÜCRE 8 — Fine-Tuned modeli Drive'a yedekle
import shutil, os
model_dir = '/content/finetuned_Qwen2.5-1.5B'
dest = '/content/drive/MyDrive/kap_finqa/finetuned_Qwen2.5-1.5B'
if os.path.exists(model_dir):
    shutil.copytree(model_dir, dest, dirs_exist_ok=True)
    print('✅ Model Drive'a kaydedildi')
else:
    print('❌ Model klasörü bulunamadı')

In [ ]:
# HÜCRE 9 — Fine-Tune: TinyLlama-1.1B (seed=42, ~1.3 saat)
!python /content/2_finetune.py \
    --model TinyLlama-1.1B \
    --train_csv /content/drive/MyDrive/kap_finqa/train.csv \
    --val_csv /content/drive/MyDrive/kap_finqa/validation.csv \
    --seed 42

In [ ]:
# HÜCRE 10 — TinyLlama modelini Drive'a yedekle
import shutil, os
model_dir = '/content/finetuned_TinyLlama-1.1B'
dest = '/content/drive/MyDrive/kap_finqa/finetuned_TinyLlama-1.1B'
if os.path.exists(model_dir):
    shutil.copytree(model_dir, dest, dirs_exist_ok=True)
    print('✅ Model Drive'a kaydedildi')
else:
    print('❌ Model klasörü bulunamadı')

## Fine-Tuned Değerlendirme

In [ ]:
# HÜCRE 11 — Eval: Qwen2.5-1.5B fine-tuned
!python /content/3_eval_finetuned.py \
    --model Qwen2.5-1.5B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv

In [ ]:
# HÜCRE 12 — Eval: TinyLlama-1.1B fine-tuned
!python /content/3_eval_finetuned.py \
    --model TinyLlama-1.1B \
    --test_csv /content/drive/MyDrive/kap_finqa/test.csv

In [ ]:
# HÜCRE 13 — Sonuçları Drive'a kaydet
import shutil, os
for fname in ['eval_finetuned_Qwen2.5-1.5B.json', 'eval_finetuned_TinyLlama-1.1B.json',
              'zero_shot_Qwen2.5-1.5B.json', 'zero_shot_TinyLlama-1.1B.json',
              'zero_shot_Mistral-7B.json']:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/kap_finqa/{fname}')
        print(f'✅ {fname} kaydedildi')